# Capa A - Clima Diario

In [2]:
import pandas as pd

# Leer únicamente la hoja Promedios_diarios
tolima = pd.read_excel(
    r"C:\Users\s.alvarezg\OneDrive - Universidad de los Andes\Documentos\Universidad\Maestria\2026-1\Ingefin\Proyecto\Tolima\Tolima_Completo.xlsx",
    sheet_name="Promedios_diarios"
)

casanare = pd.read_excel(
    r"C:\Users\s.alvarezg\OneDrive - Universidad de los Andes\Documentos\Universidad\Maestria\2026-1\Ingefin\Proyecto\Casanare\Casanare_Completo.xlsx",
    sheet_name="Promedios_diarios"
)


# Eliminar TODAS las columnas relacionadas con departamento
tolima = tolima.loc[:, ~tolima.columns.str.lower().str.contains("departamento")]
casanare = casanare.loc[:, ~casanare.columns.str.lower().str.contains("departamento")]

# Crear una sola columna departamento
tolima.insert(0, "departamento", "Tolima")
casanare.insert(0, "departamento", "Casanare")

# Unir una debajo de la otra
df_CapaA = pd.concat(
    [tolima, casanare],
    ignore_index=True,
    sort=False
)
df_CapaA  = df_CapaA .drop(columns=["departamento"])
# Guardar si quieres
df_CapaA .to_excel(
    "Capa A.xlsx",
    index=False
)

df_CapaA.head()

,department,zone,date,t_min,t_max,t_mean,precip,source
0,Tolima,Centro,2010-01-01,19.954775,0.72,10.337388,0.72,ideam
1,Tolima,Centro,2010-01-02,19.865986,0.00,9.932993,0.00,ideam
2,Tolima,Centro,2010-01-03,19.791672,0.00,9.895836,0.00,ideam
3,Tolima,Centro,2010-01-04,18.954878,0.01,9.482439,0.01,ideam
4,Tolima,Centro,2010-01-05,19.636873,0.43,10.033437,0.43,ideam


# Capa B - índice ENSO Mensual

In [13]:
import pandas as pd

df_raw = pd.read_excel("Tabla RONI.xlsx")

months = ["DJF","JFM","FMA","MAM","AMJ","MJJ","JJA","JAS","ASO","SON","OND","NDJ"]

# Pasar de formato ancho a largo
roni_monthly = (
    df_raw
    .melt(
        id_vars="Year",
        value_vars=months,
        var_name="quarter",
        value_name="roni"
    )
    .assign(
        month=lambda d: d["quarter"].map({q: i + 1 for i, q in enumerate(months)}),
        year=lambda d: d["Year"].astype(int)
    )
    .assign(
        date=lambda d: pd.to_datetime(
            dict(year=d["year"], month=d["month"], day=1)
        )
    )
)

# Clasificación ENSO
def fase_enso(x):
    if pd.isna(x):
        return "NA"
    if x >= 0.5:
        return "Niño"
    if x <= -0.5:
        return "Niña"
    return "Neutro"

roni_monthly["fase"] = roni_monthly["roni"].apply(fase_enso)

# Dejar exactamente las columnas de la foto
roni_monthly = (
    roni_monthly[["year", "month", "date", "roni", "fase"]]
    .sort_values("date")
    .reset_index(drop=True)
)
# Dejar exactamente las columnas de la foto
roni_monthly = (
    roni_monthly[["year", "month", "date", "roni", "fase"]]
    .sort_values("date")
    .reset_index(drop=True)
)

# Filtrar entre 2010 y 2025
roni_monthly = roni_monthly[
    (roni_monthly["year"] >= 2010) &
    (roni_monthly["year"] <= 2025)
].reset_index(drop=True)

df_CapaB = roni_monthly

df_CapaB.head()


df_CapaB= roni_monthly
df_CapaB

,year,month,date,roni,fase
0,2010,1,2010-01-01,1.5,Niño
1,2010,2,2010-02-01,1.2,Niño
2,2010,3,2010-03-01,0.8,Niño
3,2010,4,2010-04-01,0.4,Neutro
4,2010,5,2010-05-01,-0.2,Neutro
...,...,...,...,...,...
187,2025,8,2025-08-01,-0.3,Neutro
188,2025,9,2025-09-01,-0.4,Neutro
189,2025,10,2025-10-01,-0.5,Niña
190,2025,11,2025-11-01,-0.6,Niña


# Capa C - Rendimientos Semestrales 

In [14]:
import pandas as pd
from pathlib import Path
from functools import reduce
import zipfile

ruta_zip = r"C:\Users\s.alvarezg\OneDrive - Universidad de los Andes\Documentos\Universidad\Maestria\2026-1\Ingefin\Proyecto\Rendimientos Semestrales.zip"

carpeta_extraida = Path("Rendimientos_Semestrales_extraido")

with zipfile.ZipFile(ruta_zip, "r") as zip_ref:
    zip_ref.extractall(carpeta_extraida)

archivos = list(carpeta_extraida.rglob("*.xlsx"))

mapa_variable = {
    "Sembrado": "area_sembrada",
    "Cosechada": "area_cosechada",
    "Produccion": "produccion",
    "Rendimiento": "rendimiento"
}

def leer_archivo(ruta):
    nombre = ruta.stem

    variable = None
    for clave, valor in mapa_variable.items():
        if clave.lower() in nombre.lower():
            variable = valor
            break

    if variable is None:
        return None, None

    if "riego" in nombre.lower():
        sistema = "riego"
    elif "secano" in nombre.lower():
        sistema = "secano"
    else:
        sistema = None

    df = pd.read_excel(ruta, header=1)
    df = df.rename(columns={df.columns[0]: "periodo"})

    df = df.drop(
        columns=[c for c in df.columns if "TOTAL" in str(c).upper()],
        errors="ignore"
    )

    df_largo = df.melt(
        id_vars="periodo",
        var_name="zone",
        value_name=variable
    )

    df_largo[variable] = pd.to_numeric(df_largo[variable], errors="coerce")
    df_largo = df_largo.dropna(subset=[variable])

    df_largo["periodo"] = df_largo["periodo"].astype(str).str.strip()

    df_largo["year"] = df_largo["periodo"].str.extract(r"(\d{4})")
    df_largo["semester"] = df_largo["periodo"].str.extract(r"(II|I)")

    df_largo = df_largo.dropna(subset=["year", "semester"])

    df_largo["year"] = df_largo["year"].astype(int)
    df_largo["semester"] = df_largo["semester"].map({"I": 1, "II": 2}).astype(int)

    df_largo["system"] = sistema

    df_largo["zone"] = (
        df_largo["zone"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    df_largo = df_largo[["zone", "year", "semester", "system", variable]]

    return variable, df_largo


# Guardar dataframes por variable
dfs_por_variable = {
    "area_sembrada": [],
    "area_cosechada": [],
    "produccion": [],
    "rendimiento": []
}

for archivo in archivos:
    try:
        variable, df_temp = leer_archivo(archivo)

        if df_temp is not None:
            dfs_por_variable[variable].append(df_temp)
            print(f"Leído correctamente: {archivo.name}")
        else:
            print(f"Ignorado: {archivo.name}")

    except Exception as e:
        print(f"Error leyendo {archivo.name}: {e}")


# Unir primero los archivos de la misma variable
dfs_finales = []

for variable, lista_dfs in dfs_por_variable.items():
    if lista_dfs:
        df_variable = pd.concat(lista_dfs, ignore_index=True)

        # Por si hay duplicados exactos
        df_variable = df_variable.drop_duplicates(
            subset=["zone", "year", "semester", "system"],
            keep="last"
        )

        dfs_finales.append(df_variable)


# Ahora sí hacer merge entre variables distintas
df_final = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=["zone", "year", "semester", "system"],
        how="outer"
    ),
    dfs_finales
)

df_final = df_final[
    [
        "zone",
        "year",
        "semester",
        "system",
        "area_sembrada",
        "area_cosechada",
        "produccion",
        "rendimiento"
    ]
]

df_final = df_final.sort_values(
    ["year", "semester", "system", "zone"]
).reset_index(drop=True)

# Desde 2010
df_final = df_final[df_final["year"] >= 2010].reset_index(drop=True)

df_CapaC = df_final

df_CapaC


Leído correctamente: Cosechada, Riego, 1.xlsx
Leído correctamente: Cosechada, Riego, 2.xlsx
Leído correctamente: Cosechada, Secano, 1.xlsx
Leído correctamente: Cosechada, Secano, 2.xlsx
Leído correctamente: Produccion, Riego, 1.xlsx
Leído correctamente: Produccion, Riego, 2.xlsx
Leído correctamente: Produccion, Secano, 1.xlsx
Leído correctamente: Produccion, Secano, 2.xlsx
Leído correctamente: Rendimiento, Riego, 1.xlsx
Leído correctamente: Rendimiento, Riego, 2.xlsx
Leído correctamente: Rendimiento, Secano, 1.xlsx
Leído correctamente: Rendimiento, Secano, 2.xlsx
Leído correctamente: Sembrado, Riego, 1.xlsx
Leído correctamente: Sembrado, Riego, 2.xlsx
Leído correctamente: Sembrado, Secano, 1.xlsx
Leído correctamente: Sembrado, Secano, 2.xlsx


,zone,year,semester,system,area_sembrada,area_cosechada,produccion,rendimiento
0,Bajo Cauca,2010,1,riego,1898.0,1861.0,9828.0,5.3
1,Centro,2010,1,riego,63833.0,70772.0,433276.0,6.1
2,Costa Norte,2010,1,riego,8882.0,8246.0,44804.0,5.4
3,Llanos,2010,1,riego,28583.0,19045.0,101846.0,5.3
4,Santanderes,2010,1,riego,13498.0,11581.0,57442.0,5.0
...,...,...,...,...,...,...,...,...
305,Bajo Cauca,2025,1,secano,29996.0,32298.0,129138.0,4.0
306,Centro,2025,1,secano,971.0,310.0,1669.0,5.4
307,Costa Norte,2025,1,secano,144.0,170.0,892.0,5.3
308,Llanos,2025,1,secano,283383.0,12931.0,62037.0,4.8


# Union de capas

In [15]:
import pandas as pd

# ======================
# CAPA A + CAPA B
# ======================

# Asegurar fecha
df_CapaA["date"] = pd.to_datetime(df_CapaA["date"])

# Crear llaves año y mes
df_CapaA["year"] = df_CapaA["date"].dt.year
df_CapaA["month"] = df_CapaA["date"].dt.month

# Join diario + ENSO mensual
clima_enso = df_CapaA.merge(
    df_CapaB[["year", "month", "roni", "fase"]],
    on=["year", "month"],
    how="left"
)


# ======================
# Diario → Semestral
# ======================

# Crear semestre desde mes
clima_enso["semester"] = clima_enso["month"].apply(
    lambda x: 1 if x <= 6 else 2
)

# Agregar a nivel semestral
clima_semestral = (
    clima_enso
    .groupby(
        ["department","zone","year","semester"],
        as_index=False
    )
    .agg(
        t_min=("t_min","mean"),
        t_max=("t_max","mean"),
        t_mean=("t_mean","mean"),

        # precipitación acumulada
        precip=("precip","sum"),

        # promedio ENSO semestre
        roni=("roni","mean"),

        # fase dominante
        fase=(
            "fase",
            lambda x: x.mode().iloc[0]
            if not x.mode().empty
            else None
        )
    )
)


# ======================
# CAPA CLIMA + CAPA C
# ======================

base_final = clima_semestral.merge(
    df_CapaC,
    on=["zone","year","semester"],
    how="left"
)

# ordenar
base_final = base_final.sort_values(
    ["department","year","semester","zone"]
).reset_index(drop=True)

base_final.head()

,department,zone,year,semester,t_min,t_max,t_mean,precip,roni,fase,system,area_sembrada,area_cosechada,produccion,rendimiento
0,Casanare,Llanos,2010,1,22.254125,7.560486,14.907305,1368.447901,0.495580,Niño,riego,28583.0,19045.0,101846.0,5.3
1,Casanare,Llanos,2010,1,22.254125,7.560486,14.907305,1368.447901,0.495580,Niño,secano,123206.0,1150.0,6560.0,5.7
2,Casanare,Llanos,2010,2,21.938622,7.587525,14.763074,1396.104606,-1.431522,Niña,riego,28866.0,28583.0,159858.0,5.6
3,Casanare,Llanos,2010,2,21.938622,7.587525,14.763074,1396.104606,-1.431522,Niña,secano,3457.0,123206.0,669184.0,5.4
4,Casanare,Llanos,2011,1,21.894551,8.325526,15.110039,1506.920282,-0.765746,Niña,riego,28078.0,28866.0,146992.0,5.1


In [18]:
with pd.ExcelWriter(
    "Modelo_Completo.xlsx",
    engine="openpyxl"
) as writer:

    # Capa A
    df_CapaA.to_excel(
        writer,
        sheet_name="CapaA_Clima",
        index=False
    )

    # Capa B
    df_CapaB.to_excel(
        writer,
        sheet_name="CapaB_ENSO",
        index=False
    )

    # Capa C
    df_CapaC.to_excel(
        writer,
        sheet_name="CapaC_Semestral",
        index=False
    )

    # Resultado final
    base_final.to_excel(
        writer,
        sheet_name="Base_Final",
        index=False
    )

print("Archivo creado: Modelo_Completo.xlsx")

Archivo creado: Modelo_Completo.xlsx
